# Ensemble Learning

**Companion notebook for the BIP ML4DS lecture.** We use a binary-classification example to inspect how ensembles create and combine predictions.

> **Dataset note:** This notebook uses scikit-learn's built-in Breast Cancer Wisconsin diagnostic dataset as a generic classification example. It is not the steel-plate anomaly-detection project data, and its computed scores are not the illustrative values in the slides.

## Learning goals

- Compare hard voting and probability averaging.
- Inspect bootstrap samples, bagging, random forests, and out-of-bag estimates.
- See how boosting adds sequential corrections.
- Construct out-of-fold predictions for a stacking meta-model.
- Compare complete candidate procedures with cross-validation, then evaluate once on reserved test data.

Run the notebook from top to bottom. The final test set remains untouched until the last section. The notebook is designed for experimentation: prompts suggest changes to try using development data.

## 1. Setup and data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.base import clone
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import (
    GradientBoostingClassifier,
    RandomForestClassifier,
    StackingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

RANDOM_STATE = 42


In [ ]:
dataset = load_breast_cancer(as_frame=True)
X = dataset.data
y = dataset.target

print(f"Observations: {len(X)}")
print(f"Features: {X.shape[1]}")
print("Target classes:", dict(enumerate(dataset.target_names)))
display(y.value_counts(normalize=True).rename("proportion").to_frame())


The classes are imbalanced, so we use **balanced accuracy** for model comparisons. It averages recall across classes. We fix the metric before comparing candidates.

First reserve a final test set. All later development and selection steps use only the development data.

In [ ]:
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

display(pd.Series({
    "development observations": len(X_dev),
    "reserved test observations": len(X_test),
}).to_frame("count"))


## 2. Voting and complementary errors

An ensemble can help when its base models make different errors. The following small example mirrors the lecture: each model makes one error, and majority voting corrects all three. The values are illustrative.

In [ ]:
toy = pd.DataFrame(
    {
        "True label": [0, 0, 0, 1, 1, 1],
        "Model 1":    [1, 0, 0, 1, 1, 1],
        "Model 2":    [0, 1, 0, 1, 1, 1],
        "Model 3":    [0, 0, 1, 1, 1, 1],
    },
    index=list("ABCDEF"),
)
model_columns = ["Model 1", "Model 2", "Model 3"]
toy["Majority vote"] = (toy[model_columns].mean(axis=1) >= 0.5).astype(int)
display(toy)
print("Accuracy of each model:",
      toy[model_columns].apply(lambda col: accuracy_score(toy["True label"], col)).to_dict())
print("Majority-vote accuracy:",
      accuracy_score(toy["True label"], toy["Majority vote"]))


Now make the errors overlap. If all base models miss the same observation, majority voting cannot correct that shared error.

In [ ]:
correlated_errors = toy[model_columns].copy()
correlated_errors.loc["A", :] = 1
correlated_errors["Majority vote"] = (
    correlated_errors[model_columns].mean(axis=1) >= 0.5
).astype(int)

display(correlated_errors)
print("Majority-vote accuracy with shared error:",
      accuracy_score(toy["True label"], correlated_errors["Majority vote"]))


Hard voting combines predicted labels. Soft voting averages predicted probabilities, so the two rules can disagree. For this binary example, use a threshold of 0.5.

In [ ]:
positive_class_probabilities = np.array([0.60, 0.60, 0.01])
hard_labels = (positive_class_probabilities >= 0.5).astype(int)
hard_vote = int(np.mean(hard_labels) >= 0.5)
soft_vote = int(positive_class_probabilities.mean() >= 0.5)

print("Base labels:", hard_labels)
print("Hard vote:", hard_vote)
print(f"Mean positive-class probability: {positive_class_probabilities.mean():.3f}")
print("Soft vote:", soft_vote)


**Reflection:** What property of the base models helped in the first example? What information does soft voting use that hard voting discards?

## 3. Bootstrap samples, bagging, and random forests

Bagging trains models on bootstrap samples: samples drawn **with replacement** from the training observations. Some rows appear multiple times; others are left out of a particular sample.

In [ ]:
toy_labels = np.array(list("ABCDE"))
rng = np.random.default_rng(RANDOM_STATE)
bootstrap_indices = rng.choice(len(toy_labels), size=len(toy_labels), replace=True)
bootstrap_labels = toy_labels[bootstrap_indices]

print("Bootstrap sample:", bootstrap_labels.tolist())
print("Distinct observations:", np.unique(bootstrap_labels).tolist())
print("Left out:", np.setdiff1d(toy_labels, np.unique(bootstrap_labels)).tolist())


For a validation demonstration, split the development data once. The reserved test set remains separate. The validation split below is still development data and may be used to compare model choices.

In [ ]:
X_fit, X_demo_val, y_fit, y_demo_val = train_test_split(
    X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=RANDOM_STATE
)

print(f"Demonstration training rows: {len(X_fit)}")
print(f"Demonstration validation rows: {len(X_demo_val)}")


Fit a collection of trees on independent bootstrap samples, then average their class probabilities. We compare this bagged predictor with one tree on the same demonstration split.

In [ ]:
def fit_bootstrap_trees(X_train, y_train, n_estimators=100, random_state=42):
    rng = np.random.default_rng(random_state)
    fitted_trees = []
    n_rows = len(X_train)

    for tree_id in range(n_estimators):
        sample_indices = rng.choice(n_rows, size=n_rows, replace=True)
        tree = DecisionTreeClassifier(random_state=random_state + tree_id)
        tree.fit(X_train.iloc[sample_indices], y_train.iloc[sample_indices])
        fitted_trees.append(tree)

    return fitted_trees


def average_tree_probabilities(trees, X_new, positive_label=1):
    probabilities = np.stack([
        tree.predict_proba(X_new)[:, list(tree.classes_).index(positive_label)]
        for tree in trees
    ])
    return probabilities.mean(axis=0)


def balanced_accuracy_from_probabilities(y_true, probabilities, threshold=0.5):
    predictions = (probabilities >= threshold).astype(int)
    return balanced_accuracy_score(y_true, predictions)


bagged_trees = fit_bootstrap_trees(X_fit, y_fit, n_estimators=100, random_state=RANDOM_STATE)
single_tree = DecisionTreeClassifier(random_state=RANDOM_STATE).fit(X_fit, y_fit)

bagging_demo = pd.DataFrame(
    {
        "training": [
            balanced_accuracy_score(y_fit, single_tree.predict(X_fit)),
            balanced_accuracy_from_probabilities(
                y_fit, average_tree_probabilities(bagged_trees, X_fit)
            ),
        ],
        "validation": [
            balanced_accuracy_score(y_demo_val, single_tree.predict(X_demo_val)),
            balanced_accuracy_from_probabilities(
                y_demo_val, average_tree_probabilities(bagged_trees, X_demo_val)
            ),
        ],
    },
    index=["Single tree", "Bagged trees"],
)
display(bagging_demo.round(3))


Random forests add feature randomness at each split. Setting `max_features=1.0` considers all features at each split and gives a bagging-like comparison; `max_features="sqrt"` considers a random subset. The difference can encourage less similar trees.

In [ ]:
forest_all_features = RandomForestClassifier(
    n_estimators=100,
    max_features=1.0,
    bootstrap=True,
    random_state=RANDOM_STATE,
    n_jobs=-1,
).fit(X_fit, y_fit)

forest_random_features = RandomForestClassifier(
    n_estimators=100,
    max_features="sqrt",
    bootstrap=True,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    oob_score=True,
).fit(X_fit, y_fit)

forest_demo = pd.DataFrame(
    {
        "training balanced accuracy": [
            balanced_accuracy_score(y_fit, forest_all_features.predict(X_fit)),
            balanced_accuracy_score(y_fit, forest_random_features.predict(X_fit)),
        ],
        "validation balanced accuracy": [
            balanced_accuracy_score(y_demo_val, forest_all_features.predict(X_demo_val)),
            balanced_accuracy_score(y_demo_val, forest_random_features.predict(X_demo_val)),
        ],
    },
    index=["All features at each split", "Random feature subset"],
)
display(forest_demo.round(3))
print(f"Random forest out-of-bag accuracy: {forest_random_features.oob_score_:.3f}")


Out-of-bag (OOB) predictions use, for each training observation, trees whose bootstrap samples left that observation out. The OOB score is a development estimate. Repeatedly using it to tune choices can make it part of model selection; it does not replace the reserved final test set.

**Try changing:** the number of trees, or `max_features`. Do the validation and OOB scores move together? Does the result depend on the random seed?

## 4. Boosting adds sequential corrections

For squared-error regression, gradient boosting can fit each new tree to the current residuals. Reproduce the lecture's four-point example: start from the target mean, fit a stump to residuals, then add a fraction of that stump's prediction.

In [ ]:
x_toy = np.array([[1], [2], [3], [4]])
y_toy = np.array([2, 4, 6, 8], dtype=float)

F0 = np.full_like(y_toy, y_toy.mean())
residual_0 = y_toy - F0

stump = DecisionTreeRegressor(max_depth=1, random_state=RANDOM_STATE)
stump.fit(x_toy, residual_0)
h1 = stump.predict(x_toy)

learning_rate = 0.5
F1 = F0 + learning_rate * h1
residual_1 = y_toy - F1

boosting_steps = pd.DataFrame({
    "x": x_toy.ravel(),
    "target": y_toy,
    "initial prediction": F0,
    "first residual": residual_0,
    "stump correction": h1,
    "updated prediction": F1,
    "new residual": residual_1,
})
display(boosting_steps)
print(f"Initial training MSE: {np.mean((y_toy - F0) ** 2):.1f}")
print(f"After one correction: {np.mean((y_toy - F1) ** 2):.1f}")


The next tree would fit `residual_1`, not the original targets. The ensemble adds scaled corrections in sequence. This example demonstrates the mechanism; it does not show that training improvement guarantees better generalization.

**Reflection:** How does this update differ from averaging independently fitted trees? What does the learning rate change?

## 5. Stacking and out-of-fold predictions

Stacking trains a meta-model on base-model predictions. For each row used to train the meta-model, generate predictions from base models that did not train on that row. The code below constructs these out-of-fold (OOF) features explicitly on the demonstration training split.

In [ ]:
base_estimators = {
    "tree probability": DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE),
    "logistic probability": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000),
    ),
}

oof_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
oof_features = np.full((len(X_fit), len(base_estimators)), np.nan)

for train_indices, heldout_indices in oof_cv.split(X_fit, y_fit):
    for column, estimator in enumerate(base_estimators.values()):
        fitted_base = clone(estimator)
        fitted_base.fit(X_fit.iloc[train_indices], y_fit.iloc[train_indices])
        oof_features[heldout_indices, column] = fitted_base.predict_proba(
            X_fit.iloc[heldout_indices]
        )[:, 1]

assert not np.isnan(oof_features).any()

oof_frame = pd.DataFrame(oof_features, columns=base_estimators.keys())
oof_frame["label"] = y_fit.to_numpy()
display(oof_frame.head())


Each row now contains the original label and predictions from held-out folds. Fit the meta-model on these OOF features. To predict the demonstration validation rows, refit each base-model type on all demonstration training rows, then pass their predictions to the meta-model.

In [ ]:
meta_model = LogisticRegression(max_iter=2000).fit(oof_features, y_fit)

refitted_base_models = {}
validation_features = np.zeros((len(X_demo_val), len(base_estimators)))

for column, (name, estimator) in enumerate(base_estimators.items()):
    fitted_base = clone(estimator).fit(X_fit, y_fit)
    refitted_base_models[name] = fitted_base
    validation_features[:, column] = fitted_base.predict_proba(X_demo_val)[:, 1]

stack_validation_predictions = meta_model.predict(validation_features)
print(
    "Stack validation balanced accuracy:",
    f"{balanced_accuracy_score(y_demo_val, stack_validation_predictions):.3f}",
)


The OOF predictions train the meta-model; they do not independently evaluate the complete stack. The refitted base models also use more observations than the fold-specific models, so evaluation should assess the complete training procedure. In the next section, `StackingClassifier` handles the internal OOF construction inside each outer cross-validation fit.

**Reflection:** Why would predictions made on the same rows used to fit a base tree be a poor choice for meta-training features?

## 6. Compare complete candidate procedures

Now compare one candidate from each family on the same development data and the same stratified folds. The stack uses its own internal folds to construct OOF meta-training features. The outer cross-validation evaluates the complete candidate procedure.

In [ ]:
selection_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE + 1)

linear_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000),
)

candidates = {
    "Decision tree": DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE),
    "Random forest": RandomForestClassifier(
        n_estimators=100,
        max_features="sqrt",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "Gradient boosting": GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=2,
        random_state=RANDOM_STATE,
    ),
    "Stacking": StackingClassifier(
        estimators=[
            ("tree", DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE)),
            ("linear", linear_model),
        ],
        final_estimator=LogisticRegression(max_iter=2000),
        stack_method="predict_proba",
        cv=inner_cv,
    ),
}


The table reports mean training and held-out fold scores. In scikit-learn, `cross_validate` calls the held-out fold result `test_score`; here those folds are validation data within the development set, not the reserved final test set.

In [ ]:
comparison_rows = []

for name, candidate in candidates.items():
    result = cross_validate(
        candidate,
        X_dev,
        y_dev,
        cv=selection_cv,
        scoring="balanced_accuracy",
        return_train_score=True,
    )
    comparison_rows.append({
        "candidate": name,
        "mean training score": result["train_score"].mean(),
        "mean validation score": result["test_score"].mean(),
        "validation score SD": result["test_score"].std(ddof=1),
    })

comparison = pd.DataFrame(comparison_rows).set_index("candidate")
display(comparison.sort_values("mean validation score", ascending=False).round(3))


In [ ]:
ax = comparison[["mean training score", "mean validation score"]].plot(
    kind="bar", figsize=(9, 4.5), ylim=(0.5, 1.02), rot=15
)
ax.set_ylabel("Balanced accuracy")
ax.set_title("Three-fold cross-validation on development data")
ax.legend(frameon=False)
plt.tight_layout()


Select using the stated metric and development results. A small difference in mean scores may not indicate a meaningful difference in future performance. The test set has not been used for this choice.

In [ ]:
best_name = comparison["mean validation score"].idxmax()
best_candidate = candidates[best_name]
print("Selected candidate:", best_name)
print(f"Mean CV validation balanced accuracy: {comparison.loc[best_name, 'mean validation score']:.3f}")


### Optional explorations — do these before the final test evaluation

- Change the number of trees or `max_features` in the random forest.
- Try another learning rate or tree depth for gradient boosting.
- Change the base models or meta-model in the stack.
- Add candidates and compare them with the same folds and metric.
- Inspect score variability across folds, not only the mean.

If these changes affect the candidate set or model choices, repeat selection using development data. Do not choose among candidates using repeated test scores.

## 7. Refit and evaluate once on the reserved test set

After fixing the candidate, fit a fresh model on all development data. Evaluate it once on the untouched test set. Report the selection score and test score separately.

In [ ]:
final_model = clone(best_candidate).fit(X_dev, y_dev)
test_predictions = final_model.predict(X_test)
final_test_score = balanced_accuracy_score(y_test, test_predictions)

print("Selected candidate:", best_name)
print(f"Mean CV validation balanced accuracy: {comparison.loc[best_name, 'mean validation score']:.3f}")
print(f"Reserved test balanced accuracy: {final_test_score:.3f}")


In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_predictions,
    display_labels=dataset.target_names,
    cmap="Blues",
    values_format="d",
)
plt.title("Selected ensemble procedure on the reserved test set")
plt.tight_layout()


The test result assesses the selected and refitted procedure on this split. If test feedback leads to a new candidate choice or other development decision, the test has become part of selection and no longer provides an untouched final evaluation.

## 8. Check your understanding

1. Why can different base-model errors make voting useful?
2. What changes when a random forest considers a random subset of features at each split?
3. Why does each stacking row need predictions from models that did not train on that row?
4. Which data determine the winning candidate in the cross-validation comparison?
5. What does the reserved test score assess, and what should you do if you use it to change the model?
6. For the steel-plate project, what split would match the intended deployment setting? What metadata would you need to decide?